# Person identification

In [ ]:
import numpy as np
import torch
import os
import shutil
import glob

from models.DopplerDataset import * 
from models.Architectures import *
from utilities.training import *
from utilities.plotting import *
from utilities.data_processing import *

%load_ext autoreload
%autoreload 2

In [ ]:
# import os
# import glob
# folder_name = "doppler_traces/S1*"
# activities = []

# for folder in glob.glob(folder_name):
#     for acquisition in os.listdir(folder):
#         parts = acquisition.split("_")
#         if len(parts) > 1:
#             activity = parts[1]
#             if activity not in activities:
#                 activities.append(activity)

# print(activities)

IDEA:
- Training with datasets S1 (P1) and S3 (P2) (Training-test split)
- Test with S4 and S5 

May be valid? The problem is that from training to test, there is a change environment/monitor position. Be aware of input covariance shift

In [ ]:
# Remember to delete last time's dataset
shutil.rmtree("doppler_traces_person_identification_train") if os.path.exists("doppler_traces_person_identification_train") else None
shutil.rmtree("doppler_traces_person_identification_test") if os.path.exists("doppler_traces_person_identification_test") else None

In [ ]:
# Creating dataset for person identification task
DEBUG_MODE                 = True

TRAINING_TEST_DATASET_PATH  = "doppler_traces/S[1,3]a"
TEST_DATASET_PATH   = "doppler_traces/S[2,5]*"
DS_NAME = "person_identification"
DOPPLER_TRACE_SIZE  = 340

person_key = {'S1':1, 'S2':1, 'S3':2, 'S4':1, 'S5':2, 'S6':1, 'S7':3} # Maps dataset to corresponding person

LABELS     = ['W', 'H', 'J', 'C', 'S', 'L', 'E', 'R']
ACTIVITIES = ['Walking', 'Arm Exercises', 'Jumping', 'Sit Down/Stand Up', 'Standing', 'Sitting', 'Empty', 'Running']
PERSON_LABELS = [1, 2]
person_labels_map = {
    person: index
    for index, person in enumerate(PERSON_LABELS)
}

create_train_test_split(
    dataset_path=TRAINING_TEST_DATASET_PATH,
    doppler_trace_size=DOPPLER_TRACE_SIZE,
    activity_list=LABELS,
    train_ratio=0.6,
    ds_name=DS_NAME,
    seed=0
)

# Count number of samples from each person in the training and test datasets
train_samples = [0 for person in PERSON_LABELS]
test_samples = [0 for person in PERSON_LABELS]

for folder in glob(TRAINING_TEST_DATASET_PATH):
    for acquisition in os.listdir(folder):
        parts = acquisition.split("_")
        folder = parts[0][0:2]
        if folder == "S1":
            train_samples[0] += 1
        elif folder == "S3":
            train_samples[1] += 1

for folder in glob(TEST_DATASET_PATH):
    for acquisition in os.listdir(folder):
        parts = acquisition.split("_")
        folder = parts[0][0:2]
        if folder == "S2":
            test_samples[0] += 1
        elif folder == "S5":
            test_samples[1] += 1

print("Training samples:")
for i, count in enumerate(train_samples):
    print(f"  Person P{PERSON_LABELS[i]}: {count}")

print("Test samples:")
for i, count in enumerate(test_samples):
    print(f"  Person {PERSON_LABELS[i]}: {count}")


In [ ]:
def person_label_from_filename(filename):
    source_set = filename[:2]  # Returns the first two characters of the filename, which indicate the source set (e.g., 'S1', 'S2', etc.)
    return person_key[source_set]

batch_size = 128
augmentation = "hv"
train_dataset = DopplerDataset(
    f"doppler_traces_{DS_NAME}_train",
    person_labels_map,
    db_conversion=True,
    normalization=False,
    augmentation=augmentation,
    label_function=person_label_from_filename,
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = DopplerDataset(
    f"doppler_traces_{DS_NAME}_test",
    person_labels_map,
    db_conversion=True,
    normalization=False,
    label_function=person_label_from_filename,
)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

train_dataset.getInfo()
test_dataset.getInfo()

## SHARP (without BN)

In [ ]:
# Define SHARP model (changed)
batchNorm = False
sharp_model_noBN = SHARP(n_features=len(PERSON_LABELS), batchNorm=batchNorm)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")
learning_rate = 5e-5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model_noBN.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(sharp_model_noBN.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

In [ ]:
# Training of the model
epochs = 50
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model_noBN, 
                                                            train_dataloader, 
                                                            test_dataloader, 
                                                            epochs, 
                                                            loss_fn, 
                                                            optimizer, 
                                                            device, 
                                                            verbosity=False, 
                                                            PI=True)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, classification_report

device = torch.device("cpu")
sharp_model_noBN.to(device)

y_true = test_dataset.labels.numpy()

with torch.no_grad():
    x = test_dataset.dataset.unsqueeze(1).to(device)
    y_pred = sharp_model_noBN(x).argmax(dim=1).cpu().numpy()

print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["P1","P2"]))

In [ ]:
# Plot loss and accuracy
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, no batch normalization")

In [ ]:
# Results (metrics and plot cm)
person_labels = ["P1", "P2"]
metrics = compute_metrics("all", sharp_model_noBN, test_dataset, person_labels, device, False)
plot_confusion_matrix(metrics['cm'], person_labels)

In [ ]:
plot_f1_score(metrics['precision'], metrics['recall'], metrics['f1'], person_labels)

## SHARP (with BN)

In [ ]:
# Define SHARP model (changed)
batchNorm = False
sharp_model_noBN = SHARP(n_features=len(PERSON_LABELS), batchNorm=batchNorm)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")
learning_rate = 5e-5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model_noBN.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(sharp_model_noBN.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

In [ ]:
# Training of the model
epochs = 50
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model_noBN, 
                                                            train_dataloader, 
                                                            test_dataloader, 
                                                            epochs, 
                                                            loss_fn, 
                                                            optimizer, 
                                                            device, 
                                                            verbosity=False, 
                                                            PI=True)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, classification_report

device = torch.device("cpu")
sharp_model_noBN.to(device)

y_true = test_dataset.labels.numpy()

with torch.no_grad():
    x = test_dataset.dataset.unsqueeze(1).to(device)
    y_pred = sharp_model_noBN(x).argmax(dim=1).cpu().numpy()

print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["P1","P2"]))

In [ ]:
# Plot loss and accuracy
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, no batch normalization")

In [ ]:
# Results (metrics and plot cm)
person_labels = ["P1", "P2"]
metrics = compute_metrics("all", sharp_model_noBN, test_dataset, person_labels, device, False)
plot_confusion_matrix(metrics['cm'], person_labels)